## Setup a camera configuration for processing a single video
To process a video, a camera configuration is needed. The camera configuration makes the processing aware how to project the movie's frames to an orthorectified plane with real-world distances, and a user defined area of interest and processing resolution. It also tells the processing what the water level during the survey video is, so that the depth can be estimated, once bathymetry cross-sections are added. The process for a fixed video setup that takes videos with changing water level conditions is slightly more advanced. Therefore we here start with the assumption that you walk to a stream with a smart phone and a GPS (RTK) device or a spirit level instrument, record control points and record a short video for just one single observation.

In this notebook, we will extract one frame from the survey video to grab the control points. For this example, field observations were collected at the Ngwerere River, in Lusaka. We will first setup an empty camera configuration, and then gradually fill this with the required information. Along the way we plot what we have in a geospatial plot. The information we add is:
* Ground-control points (row and columns locations in the frame as well as real world coordinates)
* Row and column coordinates that define the area of interest.
* The water level during the video and survey (set at zero, because this survey was only done for one single video, this is only relevant if multiple videos with different water levels are processed)
* The position of the camera lens. This is relevant in case multiple videos with different water levels are processed. We here add this, but it is not actually used. 


In [ ]:
import xarray as xr
import pyorc
import matplotlib.pyplot as plt
import time
from dask.diagnostics import ProgressBar
import cv2 as cv2
from os.path import exists
import math
doss = "riverapp/Noirath/"

In [ ]:
# clear the video to take only 1 frame on 3
pass
if not exists(doss + 'video_samp.mp4'):
    
    cap = cv2.VideoCapture(doss + 'Vidéo première utilisation_crop.mp4')
    # For streams:
    #   cap = cv2.VideoCapture('rtsp://url.to.stream/media.amqp')
    # Or e.g. most common ID for webcams:
    #   cap = cv2.VideoCapture(0)
    count = 0
    img_array = []
    freq_samp = 5
    
    while cap.isOpened():
        ret, frame = cap.read()

        if ret:
            #cv2.imwrite('../../frames/frame{:d}.jpg'.format(count), frame)
            img_array.append(frame)
            height, width, layers = frame.shape
            size = (width,height)
            count += freq_samp # skip to 3 frames later
            cap.set(cv2.CAP_PROP_POS_FRAMES, count)
        else:
            cap.release()
            break

    #out = cv2.VideoWriter('riverapp/video_3_frame.mp4',cv2.VideoWriter_fourcc(*'mp4v'), 0, size)
    out = cv2.VideoWriter(doss + 'video_samp.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 25/freq_samp, size)

    for i in range(len(img_array)):
        out.write(img_array[i])
    out.release()

## Open movie and plot the first frame
We use the pyorc Video class to open a video file and extract frame number #0 (remember, python starts counting at zero instead of one). Several markers have been placed, some as square shaped checkerboard patterns, others spraypainted with black paint on a rock. All markers are more or less at the water level. If you want to interactively view coordinates you can add `%matplotlib notebook` to the top of the cell. You can then hover over the image with your mouse and see the coordinates in the bottom-right. velocimetry is normally done on one image channel (greyscale), but we first explicitly use `method="rgb"` to extract one frame in rgb colorspace for finding the points.

In [ ]:
start_sec = 7
end_sec = 12
fps_init = 25
freq_sample = 1
fps = fps_init / freq_sample 
freq = 1
start_frame = round(start_sec * fps / freq)
end_frame = round(end_sec * fps / freq)
#doss = 'riverapp/VGC1/'
file_vid = doss + 'Vidéo première utilisation.mp4'

In [ ]:
cap = cv2.VideoCapture(file_vid)
length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print( length )

In [ ]:
# uncomment line below if you want to view coordinates interactively
#%matplotlib notebook
video_file = file_vid
video = pyorc.Video(video_file, start_frame=start_frame, end_frame=end_frame)  # we only need one frame
frame = video.get_frame(0, method="rgb")


In [ ]:
def capture_event(event, x, y, falgs, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"({x}, {y})")
        src.append([x, y])

        
# plot frame on a notebook-style window
f = plt.figure(figsize=(10, 6))
cv2.imshow('image', frame)
src = []
global src
if 'flagpassed' not in locals():
    cv2.setMouseCallback('image', capture_event)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    flagpassed = True

You can identify different marker x (column) and y (row) positions in the camera's objective. Below, we have put several of these into a "src" part of the required gcp dictionary. Then we plot the frame and coordinates together.

In [ ]:
%matplotlib inline
gcps = dict(src=src)

f = plt.figure(figsize=(16, 9))
plt.imshow(frame)
plt.plot(*zip(*gcps["src"]), "rx", markersize=20, label="Control points")
plt.legend()


Now we add the rest of the information: 

* the real world coordinates of the GCPs. These were measured using an RTK GPS unit in the Universe Transverse Mercator (UTM) 35S coordinate reference system (EPSG code 32735). We add these to the GCPs using another key called "dst".
* the water level during the survey as measured in the EPSG 32735 projection (`z_0`), which is measured by the RTK GPS unit. This is used to later compute depths from a bathymetry section, measured in the same EPSG 32735 ellipsoid vertical reference.
* the coordinate reference system (`crs`). The camera configuration then understands that everything we do is in UTM 35S. Really nice, because it makes our results geographically aware and geographical plots can be made. We add the crs to the camera configuration while setting it up.

Note that in case you have a fixed camera that regularly takes movies at different water levels, you also would need to set the following:

* the locally measured water level `h_ref` during your survey. Typically this comes from a staff gauge, that a local person reads out or a pressure gauge. For each video, a new water level must then be provided, which is used to relocate the ground control points to the right location for the new water level, and to estimate the depth over cross-sections, applied later in the process. Since we here process a single video, we don't have to worry about this. 


In [ ]:
## Distances between the fours points of reference ABCD (A top left corner then clockwise order)
##  in the following order [L_AB, L_BC, L_CD, L_DA, L_AC, L_DB]
L = [82.5, 46.8, 84.1, 45.5, 94.3, 96.2]

# Points coordinates computation 
# We consider first that P1 and P4 are vertically aligned and P4 as the  origin
P1, P4 = [0,L[3]], [0,0]

# Then we compute other coordinates using cosine law
alpha = math.acos((L[3]**2+L[5]**2-L[0]**2)/(2*L[3]*L[5]))
P2 = [L[5]*math.sin(alpha),L[5]*math.cos(alpha)]
beta = math.acos((L[3]**2+L[2]**2-L[4]**2)/(2*L[3]*L[2]))
P3 = [L[2]*math.sin(beta),L[2]*math.cos(beta)]

dst = [[P2[0], P2[1]], [P3[0], P3[1]], [P4[0], P4[1]], [P1[0], P1[1]]]
print(dst)

In [ ]:
import math
# first add our local coordinates. This MUST be in precisely the same order as the src coordinates.
gcps["dst"] = dst

# # if we would use this video as survey in video, the lines below are also needed, 
# # and proper values need to be filled in. They are now commented out.
# gcps["h_ref"] = 0.155
gcps["z_0"] = 0.44

# set the height and width
height, width = frame.shape[0:2]

# now we use everything to make a camera configuration
cam_config = pyorc.CameraConfig(height=height, width=width, gcps=gcps, lens_position=[7, -2, 3])


Finally we add information to define our area of interest, and how the camera objective must be reprojected and the resolution of the velocimetry. 
* For the area of interest, 4 coordinates must be selected in the camera perspective. A geographically rectangular box will be shaped around those to make a suitable area of interest. We can simply use pixel (column row) xy coordinates for this, so we can select them using the original frame. Below, 4 points are selected and shown in the camera objective.
* a target resolution (in meters) must be selected. The resolution is used to reproject the camera objective to a planar projection with real-world coordinates.
* a window size (in number of pixels) is needed. Velocimetry will be performed in windows of this size. Since the stream is quite small, we use 1 centimeter (0.01 m) and a 25 pixel (so 25 centimeter) window size, used to find patterns on the water to trace.


In [ ]:
def capture_event(event, x, y, falgs, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"({x}, {y})")
        corners.append([x, y])

# plot frame on a notebook-style window
f = plt.figure(figsize=(10, 6))
cv2.imshow('image', frame)
corners = []
global corners

if 'flagPassedB' not in locals():
    cv2.setMouseCallback('image', capture_event)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    flaPassedB = True


In [ ]:
cam_config.set_bbox_from_corners(corners)
cam_config.resolution = 0.01
cam_config.window_size = 25

In [ ]:

f = plt.figure(figsize=(10, 6))
plt.imshow(frame)
plt.plot(*zip(*gcps["src"]), "rx", markersize=20, label="Control points")
plt.plot(*zip(*corners), "co", label="Corners of AOI")
plt.legend()


Now that all information is entered, we show the final camera configuration as a plot, both in geographical projection and in camera perspective. The rectangular box can be clearly seen now.

In [ ]:
%matplotlib inline
f = plt.figure()
ax2 = plt.axes()
ax2.imshow(frame)
cam_config.plot(ax=ax2, camera=True)

plt.savefig(doss + "ngwerere_camconfig.jpg", bbox_inches="tight", dpi=72)

Our camera configuration is ready. Below we still show a string representation and then we store the configuration to a file for use in our next notebook using the `.to_file` method.

In [ ]:
cam_config.to_file(doss + "cam_config.json")

In [ ]:
cam_config = pyorc.load_camera_config(doss + "cam_config.json")
video_file = file_vid
# video at 25 frame/second, we want from 7s to 10s
video = pyorc.Video(video_file, camera_config=cam_config, start_frame=start_frame, end_frame=end_frame, stabilize="fixed", h_a=0.)
#video

In [ ]:
da = video.get_frames()
da_norm = da.frames.normalize()
da_norm_proj = da_norm.frames.project()
piv = da_norm_proj.frames.get_piv()
delayed_obj = piv.to_netcdf(doss + "ngwerere_piv.nc", compute=False)
with ProgressBar():
    results = delayed_obj.compute()

In [ ]:
import xarray as xr
import pyorc
from matplotlib.colors import Normalize
cap = cv2.VideoCapture(file_vid)
length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print( length )

In [ ]:
ds = xr.open_dataset(doss + "ngwerere_piv.nc")
# first re-open the original video, extract one RGB frame and plot that
video_file = file_vid

video = pyorc.Video(video_file, start_frame=start_frame, end_frame=end_frame)
# borrow the camera config from the velocimetry results
video.camera_config = ds.velocimetry.camera_config

da_rgb = video.get_frames(method="rgb")
# project the rgb frame
da_rgb_proj = da_rgb.frames.project()
# plot the first frame (we only have one) without any arguments, default is to use "local" mode
p = da_rgb_proj[0].frames.plot()

# now plot the results on top, we use the mean, because we cannot plot more than 2 dimensions. 
# Default plotting method is "quiver", but "scatter" or "pcolormesh" is also possible.
# We add a nice colorbar to understand the magnitudes.
# We give the existing axis handle of the mappable returned from .frames.plot to plot on, and use 
# some transparency.
ds_mean = ds.mean(dim="time", keep_attrs=True)

# first a pcolormesh
ds_mean.velocimetry.plot.pcolormesh(
    ax=p.axes,
    alpha=0.3,
    cmap="rainbow",
    add_colorbar=True,
    vmax=0.6
)

ds_mean.velocimetry.plot(
    ax=p.axes,
    color="w",
    alpha=0.5,
    width=0.0015,
)

In [ ]:
import copy
ds_mask = copy.deepcopy(ds)
mask_corr = ds_mask.velocimetry.mask.corr(inplace=True)
mask_minmax = ds_mask.velocimetry.mask.minmax(inplace=True)
mask_rolling = ds_mask.velocimetry.mask.rolling(inplace=True)
mask_outliers = ds_mask.velocimetry.mask.outliers(inplace=True)
mask_var = ds_mask.velocimetry.mask.variance(inplace=True)
mask_angle = ds_mask.velocimetry.mask.angle(inplace=True)
mask_count = ds_mask.velocimetry.mask.count(inplace=True)


# apply the plot again, let's leave out the scalar values, and make the quivers a bit nicer than before.
ds_mean_mask = ds_mask.mean(dim="time", keep_attrs=True)


# again the rgb frame first
p = da_rgb_proj[0].frames.plot()

#...and then masked velocimetry
ds_mean_mask.velocimetry.plot(
    ax=p.axes,
    alpha=0.4,
    cmap="rainbow",
    scale=20,
    width=0.0015,
    norm=Normalize(vmax=0.6, clip=False),
    add_colorbar=True
)


In [ ]:
# apply all methods in time domain with relaxed angle masking
import numpy as np
ds_mask2 = copy.deepcopy(ds)
ds_mask2.velocimetry.mask.corr(inplace=True)
ds_mask2.velocimetry.mask.minmax(inplace=True)
ds_mask2.velocimetry.mask.rolling(inplace=True)
ds_mask2.velocimetry.mask.outliers(inplace=True)
ds_mask2.velocimetry.mask.variance(inplace=True)
ds_mask2.velocimetry.mask.angle(angle_tolerance=0.5*np.pi)
ds_mask2.velocimetry.mask.count(inplace=True)
ds_mask2.velocimetry.mask.window_mean(wdw=2, inplace=True, tolerance=0.5, reduce_time=True)

# Now first average in time before applying any filter that only works in space.
ds_mean_mask2 = ds_mask2.mean(dim="time", keep_attrs=True)



# apply the plot again
# again the rgb frame first
p = da_rgb_proj[0].frames.plot()

#...and then filtered velocimetry
ds_mean_mask2.velocimetry.plot(
    ax=p.axes,
    alpha=0.4,
    cmap="rainbow",
    scale=20,
    width=0.0015,
    norm=Normalize(vmax=0.6, clip=False),
    add_colorbar=True
)


In [ ]:
# again the rgb frame first, but now the unprojected one. Now we use the "camera" mode to plot the camera perspective
p = da_rgb[0].frames.plot(mode="camera")

#...and then masked velocimetry again, but also camera. This gives us an augmented reality view. The quiver scale 
# needs to be adapted to fit in the screen properly
ds_mean_mask2.velocimetry.plot(
    ax=p.axes,
    mode="camera",
    alpha=0.4,
    cmap="rainbow",
    scale=200,
    width=0.0015,
    norm=Normalize(vmin=0., vmax=0.6, clip=False),
    add_colorbar=True
)


In [ ]:
ds_mask2.velocimetry.set_encoding()
ds_mask2.to_netcdf(doss + "ngwerere_masked.nc")

In [ ]:
import xarray as xr
import pyorc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import time
import json
from dask.diagnostics import ProgressBar
import cv2 as cv2
from os.path import exists
import math
import pandas as pd
#doss = "riverapp/Test1/"
#start_sec = 25
#end_sec = 30
#fps = 8
#freq = 1
#start_frame = round(start_sec * fps / freq)
#end_frame = round(end_sec * fps / freq)
#file_vid = doss + 'video_samp.mp4'

In [ ]:
ds = xr.open_dataset(doss + "ngwerere_masked.nc")

# also open the original video file
video_file = file_vid
video = pyorc.Video(video_file, start_frame=start_frame, end_frame=end_frame)
# borrow the camera config from the velocimetry results
video.camera_config = ds.velocimetry.camera_config
# get the frame as rgb
da_rgb = video.get_frames(method="rgb")
frame = video.get_frame(0, method="rgb")

In [ ]:
def capture_event(event, x, y, falgs, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"({x}, {y})")
        C.append([x, y])
        
# plot frame on a notebook-style window
f = plt.figure(figsize=(10, 6))
cv2.imshow('image', frame)
C = []
global C
if 'flagpassedC' not in locals():
    cv2.setMouseCallback('image', capture_event)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    flagpassed = True

In [ ]:
f = open(doss + 'cam_config.json')
gcps = json.load(f)['gcps']
f.close()
src = np.float32(gcps['src'])
dst = np.float32(gcps['dst'])
M = np.array(cv2.getPerspectiveTransform(src, dst))
print(M)
trans_points = cv2.perspectiveTransform(np.float32([C]), M)[0]
print(trans_points)
print('##############')

cam_config = pyorc.load_camera_config(doss + "cam_config.json")
M = cam_config.get_M(reverse=False, h_a=0.44)
print(M)
trans_points = cv2.perspectiveTransform(np.float32([C]), M)[0]
print(trans_points)


In [ ]:
noirath = pd.read_csv(doss + "bathy_format_riverApp.txt")
x_bath = noirath["x"].values
z_bath = noirath["y"].values


#pyorc.Transect.get_xyz_perspective(M=M, xs=x_bath, ys=z_bath)
C1, C2, C3, C4 = trans_points


P2, P3, P4, P1 = [[6.646197057479179, 7.605134099616857], [6.689990026724214, 0.011551724137932139], [0, 0], [0, 7.83]]
water_level = 0.29

x, y, z = np.empty((3, len(x_bath)))
print(np.amax(x_bath))
print(np.amax(z_bath))
for idx, xb in enumerate(x_bath):
    lambd = xb / np.amax(x_bath)
    z_max = np.amax(z_bath)
    x[idx] = C1[0] + lambd * (C2[0] - C1[0])
    y[idx] = C1[1] + lambd * (C2[1] - C1[1])
    #y[idx] = C1[1]
    #z[idx] = (z_bath[idx]/z_max) * water_level
    z[idx] = z_max - z_bath[idx]
    
    
x2, y2, z2 = np.empty((3, len(x_bath)))

for idx, xb in enumerate(x_bath):
    lambd = xb / np.amax(x_bath)
    z_max = np.amax(z_bath)
    x2[idx] = C3[0] + lambd * (C4[0] - C3[0])
    y2[idx] = C3[1] + lambd * (C4[1] - C3[1])
    z2[idx] = (z_bath[idx]/z_max) * water_level
    

bathy = pd.read_csv(doss + "bathy.dat", delim_whitespace=True)
x2 = bathy["x"].values
y2 = bathy["y"].values
z_min = np.amin(bathy["z"].values)
z2 = bathy["z"].values - z_min
print(z2)
        

In [ ]:
#ds_points = ds.velocimetry.get_transect(x, y, z, rolling=4)
ds_points2 = ds.velocimetry.get_transect(x2, y2, z2, rolling=4)

In [ ]:
#ds_points_q = ds_points.transect.get_q(v_corr=0.85, fill_method="log_interp")
ds_points_q2 = ds_points2.transect.get_q(v_corr=0.85, fill_method="log_interp")

In [ ]:
ax = plt.axes()
#ds_points_q["v_eff"].isel(quantile=2).plot(ax=ax, label="q")
ds_points_q2["v_eff"].isel(quantile=2).plot(ax=ax, label="q2")
plt.legend()
plt.grid()

In [ ]:
# plot the rgb frame first. We use the "camera" mode to plot the camera perspective.
norm = Normalize(vmin=0., vmax=0.8, clip=False)

p = da_rgb[0].frames.plot(mode="camera")

# extract mean velocity and plot in camera projection
ds.mean(dim="time", keep_attrs=True).velocimetry.plot(
    ax=p.axes,
    mode="camera",
    cmap="rainbow",
    scale=200,
    width=0.001,
    alpha=0.3,
    norm=norm,
)

# plot velocimetry point results in camera projection
# ds_points_q.isel(quantile=2).transect.plot(
#     ax=p.axes,
#     mode="camera",
#     cmap="rainbow",
#     scale=100,
#     width=0.003,
#     norm=norm,
# )
ds_points_q2.isel(quantile=2).transect.plot(
    ax=p.axes,
    mode="camera",
    cmap="rainbow",
    scale=100,
    width=0.003,
    norm=norm,
    add_colorbar=True
)

# store figure in a JPEG
p.axes.figure.savefig("ngwerere.jpg", dpi=200)

In [ ]:
# again plot the projected background
from matplotlib.colors import Normalize
norm = Normalize(vmin=0, vmax=0.6, clip=False)
ds_mean = ds.mean(dim="time", keep_attrs=True)
p = da_rgb.frames.project()[0].frames.plot(mode="local")

# plot velocimetry point results in local projection
# ds_points_q.isel(quantile=2).transect.plot(
#     ax=p.axes,
#     mode="local",
#     cmap="rainbow",
#     scale=10,
#     width=0.003,
#     norm=norm,
#     add_colorbar=True,
# )

ds_points_q2.isel(quantile=2).transect.plot(
    ax=p.axes,
    mode="local",
    cmap="rainbow",
    scale=10,
    width=0.003,
    norm=norm,
    add_colorbar=True,
)
# to ensure streamplot understands the directions correctly, all values must 
# be flipped upside down and up-down velocities become down-up velocities.
ds_mean.velocimetry.plot.streamplot(
    ax=p.axes,
    mode="local",
    density=3.,
    minlength=0.05,
    linewidth_scale=2,
    cmap="rainbow",
    norm=norm,
    add_colorbar=True
)


In [ ]:
# ds_points_q.transect.get_river_flow()
# print(ds_points_q["river_flow"])
ds_points_q2.transect.get_river_flow()
print(ds_points_q2["river_flow"])
